In [ ]:
from dotenv import load_dotenv
from langchain_classic.chat_models import init_chat_model
from langchain_classic.document_loaders import PyPDFLoader
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: str | ForwardRef('os.PathLike[str]') | None = None, stream: IO[str] | None = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: str | None = 'utf-8') -> bool>

In [21]:
import sys
from pathlib import Path

# Notebook 的工作目录通常是当前文件所在目录，向上查找服务根目录。
service_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
        and (path / "app" / "core" / "config.py").is_file()
    ),
    None,
)
if service_root is None:
    raise RuntimeError("未找到 agent-service 根目录，请从该项目或 notebooks 子目录运行。")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))


In [7]:

# 1.构建知识库
# 1.1.加载文档

loader = PyPDFLoader(file_path="resources/贵州茅台研报.pdf", mode="single")
docs = loader.load()
docs

[Document(metadata={'producer': '; modified using iTextSharp 5.2.1 (c) 1T3XT BVBA', 'creator': '', 'creationdate': '', 'author': '', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-04-27T09:57:46+08:00', 'sourcemodified': "D:20260426155317+08'00'", 'subject': '', 'title': '国信证券-贵州茅台(600519)2026Q1收入稳健增长，市场化改革成效逐步显现-260426.pdf', 'trapped': 'False', 'source': 'resources/贵州茅台研报.pdf', 'total_pages': 6}, page_content='请务必阅读正文之后的免责声明及其项下所有内容\n证券研究报告 | 2026年04月25日\n优于大市贵州茅台（600519.SH）\n2026Q1 收入稳健增长，市场化改革成效逐步显现\n核心观点 公司研究·财报点评\n食品饮料·白酒Ⅱ\n证券分析师：张向伟 证券分析师：张未艾\n021-61761031\nzhangxiangwei@guosen.com.cn zhangweiai@guosen.com.cn\nS0980523090001 S0980525070005\n基础数据\n投资评级 优于大市(维持)\n合理估值\n收盘价 1458.49 元\n总市值/流通市值 1826424/1826424 百万元\n52周最高价/最低价 1645.00/1322.01 元\n近3 个月日均成交额 6551.18 百万元\n市场走势\n资料来源：Wind、国信证券经济研究所整理\n相关研究报告\n《贵州茅台（600519.SH）-2025 年释放经营压力，2026 年改革\n破局》 ——2026-04-18\n《贵州茅台（600519.SH）-革故鼎新，与时偕行，再论茅台的护\n城河与增长潜力》 ——2026-03-19\n《贵州茅台（600519.SH）-全面推进市场化转型，解决供需适配\n问题》 ——2026-01-

In [8]:

# 1.2.切分文档
# 创建递归切分器
splitter = CharacterTextSplitter(separator="\n", chunk_size=800, chunk_overlap=150)
# 切分文档
chunks = splitter.split_documents(docs)
print(f"分块数量：{len(chunks)}")

分块数量：14


In [15]:
# 1.3.向量化
# 向量模型
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024, base_url="http://10.40.0.31:11434")
# 1.4.存入向量库
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)

In [16]:
# 2.在线问答阶段

model = init_chat_model(
    "deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


def try_rag(query: str):
    # 2.1.检索文档（VectorStore会自动把问题向量化，召回相关知识片段）
    retrieved_docs = vectorstore.similarity_search(query, k=2)

    # 2.2.拼接上下文提示词
    content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"""你基于我提供的报告回答用户问题，报告中没提及的就说不知道，不要自己编造答案.
    report: ```{content}```
    query: {query}"""
    
    # 2.3.调用模型，生成答案
    response = model.invoke(prompt)
    return response.content


# 3.测试
print(try_rag("茅台收盘价多少"))
print('=' * 100)
print(try_rag("茅台2025年的市盈率和市净率是多少"))

1458.49元
根据报告，茅台2025A/E的PE为22.2倍。报告中未提及市净率数据，因此无法回答市净率是多少。


In [17]:
from langchain_community.document_loaders import PyPDFLoader

# 初始化并配置文档加载器
loader = PyPDFLoader(
    file_path='resources/贵州茅台研报.pdf',
    mode="page"   # single: 整个文件作为1页文档. page：按PDF分页
)

# 加载 PDF 文件，得到 list[Document]
docs = loader.load()
print(f'文档数量:{len(docs)}')

# 遍历，得到每一个 Document，都包含 page_content 和 metadata 两个属性
for i, doc in enumerate(docs):
    print(f"==============Doc_{i}=================")
    print("metadata = ", doc.metadata)
    print("page_content = ", doc.page_content)

文档数量:6
==============Doc_0=================
metadata =  {'producer': '; modified using iTextSharp 5.2.1 (c) 1T3XT BVBA', 'creator': '', 'creationdate': '', 'author': '', 'comments': '', 'company': '', 'keywords': '', 'moddate': '2026-04-27T09:57:46+08:00', 'sourcemodified': "D:20260426155317+08'00'", 'subject': '', 'title': '国信证券-贵州茅台(600519)2026Q1收入稳健增长，市场化改革成效逐步显现-260426.pdf', 'trapped': 'False', 'source': 'resources/贵州茅台研报.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}
page_content =  请务必阅读正文之后的免责声明及其项下所有内容
证券研究报告 | 2026年04月25日
优于大市贵州茅台（600519.SH）
2026Q1 收入稳健增长，市场化改革成效逐步显现
核心观点 公司研究·财报点评
食品饮料·白酒Ⅱ
证券分析师：张向伟 证券分析师：张未艾
021-61761031
zhangxiangwei@guosen.com.cn zhangweiai@guosen.com.cn
S0980523090001 S0980525070005
基础数据
投资评级 优于大市(维持)
合理估值
收盘价 1458.49 元
总市值/流通市值 1826424/1826424 百万元
52周最高价/最低价 1645.00/1322.01 元
近3 个月日均成交额 6551.18 百万元
市场走势
资料来源：Wind、国信证券经济研究所整理
相关研究报告
《贵州茅台（600519.SH）-2025 年释放经营压力，2026 年改革
破局》 ——2026-04-18
《贵州茅台（600519.SH）-革故鼎新，与时偕行，再论茅台的护
城河与增长潜力》 ——2026-03-19
《贵州茅台

In [18]:
# 写到本地看看
with open("resources/r1.md", "w", encoding="utf-8") as f:
    f.write(docs[0].page_content)

In [ ]:
# mineru 案例
from mineru import MinerU

# 创建客户端，Flash 模式，无需 Token
client = MinerU()
# 解析文件
result = client.flash_extract("resources/sample.pdf")

# 获取 markdown
print(result.markdown)
# 保存到本地
result.save_markdown("resources/r2.md")

# The response of flow duration curves to afforestation

Patrick N.J. Lanea,c,\*, Alice E. Bestb,c,d, Klaus Hickelb,c, Lu Zhangb,c

a School of Forest and Ecosystem Studies, University of Melbourne, P.O. Box 137, Heidelberg, Victoria 3084, Australia

c Cooperative Research Centre for Catchment Hydrology, Canberra, ACT, Australia d Department of Civil and Environmental Engineering, University of Melbourne, Victoria, Australia

Received 1 October 2003; revised 22 December 2004; accepted 3 January 2005

## Abstract

The hydrologic effect of replacing pasture or other short crops with trees is reasonably well understood on a mean annual basis. The impact on flow regime, as described by the annual flow duration curve (FDC) is less certain. A method to assess the impact of plantation establishment on FDCs was developed. The starting point for the analyses was the assumption that rainfall and vegetation age are the principal drivers of evapotranspiration. A key objective was to remove the var

PosixPath('resources/r2.md')

In [34]:
from app.core.config import settings

# 创建客户端，Precision 模式，需要 Token
client = MinerU(settings.rag.mineru_token)
# 解析文件
result = client.extract("resources/new_sample.pdf")
# 获取 markdown
print(result.markdown)
# 输出到本地
result.save_markdown("resources/r3.md", with_images=True)

www.elsevier.com/locate/jhydrol

# The response of flow duration curves to afforestation

Patrick N.J. Lane<sup>a,c,\*</sup>, Alice E. Best<sup>b,c,d</sup>, Klaus Hickel<sup>b,c</sup>, Lu Zhang<sup>b,c</sup>

<sup>a</sup>School of Forest and Ecosystem Studies, University of Melbourne, P.O. Box 137, Heidelberg, Victoria 3084, Australia

<sup>b</sup>CSIRO Division of Land and Water, Canberra, ACT, Australia

<sup>c</sup>Cooperative Research Centre for Catchment Hydrology, Canberra, ACT, Australia <sup>d</sup>Department of Civil and Environmental Engineering, University of Melbourne, Victoria, Australia

Received 1 October 2003; revised 22 December 2004; accepted 3 January 2005

## Abstract

The hydrologic effect of replacing pasture or other short crops with trees is reasonably well understood on a mean annual basis. The impact on flow regime, as described by the annual flow duration curve (FDC) is less certain. A method to assess the impact of plantation establishment on FDCs was develo

PosixPath('resources/r3.md')

In [35]:
# 创建客户端，Precision 模式，需要 Token
client = MinerU(settings.rag.mineru_token)
# 解析文件，支持各种自定义参数，例如：ocr、
result = client.extract('resources/small_ocr.pdf', ocr=True)

# 获取 markdown
print(result.markdown)
# 输出到本地
result.save_markdown('resources/output/r4.md')

史的事情。(3)为有用物的量找到社会尺度, 也是这样。商品的这些尺度之所以不同, 部分是由于被计量的物的性质不同, 部分是由于约定俗成。

物的有用性使物成为使用价值。(4)但这种有用性不是飘忽不定的。它决定于商品体的属性，离开了商品体就不存在。因此，商品体本身，例如铁、小麦、金钢石等等，就是使用价值。赋予商品体以这种性质的，不是人为了取得它的有用性质所耗费的劳动的多少。在谈到使用价值时，总是指一定的量而言，如一打表，一米布，一吨铁等等。商品的使用价值为商品学和商业成规这种专门的知识提供材料。(5)使用价值只是在使用或消费中得到实现。不论财富的社会形式如何，使用价值构成财富的物质。在我们所要考察的社会形式中，使用价值同时又是交换价值的物质承担者。

交换价值首先表现为量的关系，表现为不同种使用价值彼此相交换的比例 $^{(6)}$ ，即随着时间和地点的不同而不断改变的关系。因此，交换价值好象是一种任意的、纯粹相对的东西；商品固有的、内在的交换价值似乎是经院哲学家所说的形容语的矛盾①。(7)现在我们进一步考察这个问题。

某种特殊的商品，例如一夸特小麦，按各种极不相同的比例同别的商品交换。但是，它的交换价值，无论采用何种表现方式，x 量鞋油、y 量绸缎、z 量金等等，都是不变的。因此，它必定有一种与这些不同的表现相区别的内容。

我们再拿两种商品例如小麦和铁来说。不管二者的交换比例怎样，总是可以用一个等式来表示：一定量的小麦等于若干量的铁，如1夸特小麦=a公斤铁。这个等式说明什么呢？它说明在两种不同的物里面，即在1夸特小麦和a公斤铁里面，有一种共同的东西。因而这二者都等于第三种东西，后者本身既不是第一种物，也不是第二种物。这二者中的每一个作为交换价值，都必定能不依赖另一个而化为第三种东西。

用一个初级几何的例子就可以说明这一点。为了测量和比较各种直线形的面积，就把它们分成三角形，再把三角形化成与它的外形完全不同的表现——底乘高的一半。各种商品的交换价值也同样要化成一种对它们来说是共同的东西，各自代表这种共同东西的多量或少量。

这种共同东西不可能是商品的某种天然属性，几何的、物理的、化学的属性等等。商品的天然属性只是就它们使商品有用，从而使商品成为使用价值来说，才加以考虑。但是，另一方面很清楚，商品的使用价值在商品交换时被抽象掉了，而一切商品交换关系的特点正是这种抽

PosixPath('resources/output/r4.md')

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 准备一段较长的文本
with open('resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 文本中是否携带标题
)

# 切分文档，接收str，返回值是 list[Document]
chunks = markdown_splitter.split_text(doc)

for chunk in chunks:
    print(chunk.model_dump_json(indent=2))

for i, chunk in enumerate(chunks, 1):
    chunk.id = f"doc-{i}"

{
  "id": null,
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（一）教育学萌芽时期代表作"
  },
  "page_content": "# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（二）《学记》主要思想"
  },
  "page_content": "### （二）《学记》主要思想  \n**教育作用**：化民成俗，其必由学；建国君民，教学为先。  \n**教学相长**：教和学两方面互相影响和促进，都得到提高。  \n**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  \n**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”  \n**启发诱导**：道而弗牵，强而弗抑，开而弗达。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（三）孔子及《论语》主要思想"
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、

In [37]:
from langchain_ollama import OllamaEmbeddings

# 创建Embedding模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b", 
    dimensions=1024, 
    base_url="http://10.40.0.31:11434"
)


# 向量化单条文本
text = "我爱上班"
vector = ollama_embeddings.embed_query(text)


# 批量向量化
texts = ["我要躺平", "我爱工作", "拒绝加班"]
vectors = ollama_embeddings.embed_documents(texts)
print(f"\n批量向量化: {len(vectors)} 条, 维度: {len(vectors[0])}")


print('==============衡量相似度===============')
import numpy as np


def cosine_similarity(vec1, vec2):
    """余弦相似度函数"""
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 比较向量相似度
for i, v in enumerate(vectors):
    similarity = cosine_similarity(vector, v)
    print(f"{texts[i]}:", similarity)


批量向量化: 3 条, 维度: 1024
==============衡量相似度===============
我要躺平: 0.41088322856359005
我爱工作: 0.8738842503116766
拒绝加班: 0.42182914548071127


In [40]:
# 将文档保存到向量数据库
memory_vector_store = InMemoryVectorStore(embedding=ollama_embeddings)

ids = memory_vector_store.add_documents(chunks)

print(ids)

['doc-1', 'doc-2', 'doc-3', 'doc-4', 'doc-5', 'doc-6', 'doc-7', 'doc-8', 'doc-9', 'doc-10', 'doc-11', 'doc-12', 'doc-13', 'doc-14', 'doc-15', 'doc-16', 'doc-17', 'doc-18', 'doc-19', 'doc-20', 'doc-21']


In [41]:
# 删除旧文档
memory_vector_store.delete(["doc-1", "doc-2", "doc-3"])

In [42]:
# 相似度搜索
# 用户问题
query = "论语中教育的目的是什么"

# 相似度检索，返回值是 list[Document]
results = memory_vector_store.similarity_search(
    query=query,
    k = 5,
)

# 打印结果
for doc in results:
    print(doc.model_dump_json(indent=2))

{
  "id": "doc-4",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}
{
  "id": "doc-5",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。",
  "type": "Document"
}
{
  "id": "doc-9",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第二节 教育的定义",
    "Header 3": "（一）教育的定义"
  },
  "page_content": "## 第二节 教育的定义  \n### （一）教育的定义  \n广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  \n狭义的教育就是指学校教育。  \n教育的要素：教育者、受教育者、教育影响（主要是教育内容）。",
  "type": "Document"
}
{
  "id": "doc-6",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（

In [43]:
# 带相似度得分的检索
results = memory_vector_store.similarity_search_with_score(
    query,
    k = 5,
)

# 遍历结果，
for doc, score in results:
    print(f"==========score: {score}=============")
    print(doc.model_dump_json(indent=2))

==========score: 0.6506787607292287=============
{
  "id": "doc-4",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}
==========score: 0.6200887484686738=============
{
  "id": "doc-5",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（五）荀子主要思想"
  },
  "page_content": "### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。",
  "type": "Document"
}
==========score: 0.5839384803405735=============
{
  "id": "doc-9",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第二节 教育的定义",
    "Header 3": "（一）教育的定义"
  },
  "page_content": "## 第二节 教育的定义  \n### （一）教育的定义  \n广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  \n狭义的教育就是指学校教育。  \n教育的要素：教育者、受教育者、教育影响（主要是教

In [44]:
from app.core.config import settings
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# 1.初始化模型
model = init_chat_model(
    "deepseek-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}}
)


# 2.定义工具
@tool
def retrieve_docs(query: str):
    """检索知识库，获取知识片段"""
    # 检索知识片段, list[Document]
    docs = memory_vector_store.similarity_search(query, k=2)
    # 拼接返回，不需要我们组织提示词，tool 结果会直接给模型
    return "\n\n".join(doc.page_content for doc in docs)


# 3.初始化 Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=[retrieve_docs],
    system_prompt='''
    你是一个教资考试专家，你调用工具检索文档回答用户问题，文档中没有的不要编造，直接说不知道。
    回答直接引用文档中原文，不要自己发挥！
    '''
)
# 4.调用Agent
query = "论语中教育的目的是什么？"
response = agent.invoke({'messages': [HumanMessage(content=query)]})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================
Tool Calls:
  retrieve_docs (call_00_9hqN51Q0hXpfRKsxILSZ8806)
 Call ID: call_00_9hqN51Q0hXpfRKsxILSZ8806
  Args:
    query: 论语 教育的目的
================================= Tool Message =================================
Name: retrieve_docs

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。

### （五）荀子主要思想  
**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  
**教学原则**：学以致用，锲而不舍。
================================== Ai Message ==================================

检索到的文档中没有关于《论语》教育目的的内容（返回的是孟子、荀子的思想），我不知道。


In [45]:
# 检索
query = "你好"

response = agent.invoke({'messages': [HumanMessage(content = query)]})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！我是教资考试专家，可以帮你解答教师资格证考试相关的问题。请问你想了解哪方面的内容呢？
